# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — “What Predicts Health?” (ML Appendix, Random Forest feature importance).** Average Position (43%) and Impressions (32%) are reported as the top two predictors of Health Score, together 75% of total importance.

- *Where does the label come from?* Health Score is FlyRank's own composite: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). It isn't an independent outcome — 60 of its 100 points are built directly from two of the "predictor" features being ranked.
- *Does the validation design carry the claim?* The paper is admirably upfront about this — it states the importance is "descriptive rather than causal" because the target is partly constructed from the inputs. That's the right instinct, and it's the exact same check I ran on my own baseline in `w03`/`w06` (`corr(baseline_score, avg_ctr) = -1.000`). My constructive question: since the paper already flags the overlap, how much of that 75% combined importance is compositional (guaranteed by the scoring formula) versus a genuinely discovered relationship? A useful follow-up test would be predicting only the two *smaller*-weighted components (CTR, Scroll Depth) from Position/Impressions — that would isolate the model's real discriminative signal from the part that's guaranteed by arithmetic.

**Finding 2 — Growth prediction (logistic regression, 71% holdout accuracy).** Content Age is reported as the strongest negative signal separating growing from declining pages.

- *Where does the label come from?* `trend_direction`, defined in the glossary as the 30-day-vs-previous-30-day impression change (Up: >10%, Down: >10% decline) — a real, independently computed outcome, not a circular one. Good design here.
- *Does the validation design carry the claim?* The methodology section confirms an 80/20 holdout split but doesn't say whether it's grouped by brand or random. With 57 brands in the portfolio, if pages from the same brand can land in both train and test, the model could partly be learning "which brand is currently trending" rather than a content-level growth pattern — the same grouping risk I tested directly in Section 2 above, where the naive-vs-grouped AUC gap showed exactly how much a client-level split changes the number. The paper's own Confounding Variables section already flags "Content age confounds model-performance comparisons," which makes this worth asking rather than assuming: was the 71% holdout accuracy checked against a brand-grouped split, and if not, how much of it might be brand-level momentum rather than a transferable content signal?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week-5 already used a grouped split (`GroupShuffleSplit` on `client_hash_id`), so the honest number already exists. What's shown here is the reverse direction — the same model and features, but under a naive **random** split that ignores client grouping, to make the size of the inflation visible rather than just asserting grouping matters.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"
FINAL_MONTH = "2026-06"

# --- Rebuild the exact w05 feature set ---
raw = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS avg_ctr,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS total_impressions,
        SUM(f.ga4_sessions) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS sessions_per_impression,
        SUM(f.ga4_engaged_sessions) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS engagement_rate,
        SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS ai_referral_share,
        DATE_DIFF('day', MAX(d.content_created_date), DATE '{MID_MONTH}-01') AS content_age_days,
        MAX(d.word_count) AS word_count,
        MAX(d.search_volume) AS search_volume,
        MAX(d.competition) AS competition,
        MAX(d.content_type) AS content_type,
        MAX(d.main_intent) AS main_intent
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
    WHERE c.access_profile = 'gsc_and_ga4' AND d.is_deleted IS FALSE AND d.is_published IS TRUE
    GROUP BY 1, 2
""").df()
raw = raw.dropna(subset=['avg_ctr', 'avg_position'])

ratio_cols = ['sessions_per_impression', 'engagement_rate', 'ai_referral_share']
raw[ratio_cols] = raw[ratio_cols].fillna(0)
raw['word_count'] = raw['word_count'].fillna(raw['word_count'].median())
raw['search_volume'] = raw['search_volume'].fillna(0)
raw['competition'] = raw['competition'].fillna(raw['competition'].median())
raw['content_type'] = raw['content_type'].fillna('unknown')
raw['main_intent'] = raw['main_intent'].fillna('unknown')
raw['high_performer'] = (raw['avg_ctr'] > raw['avg_ctr'].median()).astype(int)

cluster_input_cols = ['avg_position', 'total_impressions', 'sessions_per_impression', 'engagement_rate',
                       'ai_referral_share', 'content_age_days', 'word_count', 'search_volume', 'competition']
scaler = StandardScaler()
X_cluster = scaler.fit_transform(raw[cluster_input_cols])
raw['archetype_cluster'] = KMeans(n_clusters=6, random_state=0, n_init=10).fit_predict(X_cluster).astype(str)

cat_cols = ['content_type', 'main_intent', 'archetype_cluster']
encoded = pd.get_dummies(raw, columns=cat_cols, prefix=['ctype', 'intent', 'cluster'])
feature_cols_model = [c for c in encoded.columns
                       if c not in ('content_hash_id', 'client_hash_id', 'avg_ctr', 'high_performer')]

def train_eval(train_df, test_df, cols):
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0, class_weight='balanced')
    rf.fit(train_df[cols], train_df['high_performer'])
    return roc_auc_score(test_df['high_performer'], rf.predict_proba(test_df[cols])[:, 1])

# --- "Before": naive random split, ignores client grouping ---
train_naive, test_naive = train_test_split(encoded, test_size=0.3, random_state=0, stratify=encoded['high_performer'])
naive_auc = train_eval(train_naive, test_naive, feature_cols_model)
naive_overlap = set(train_naive['client_hash_id']) & set(test_naive['client_hash_id'])

# --- "After": honest grouped split, same as Week 5 ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(encoded, groups=encoded['client_hash_id']))
train_grouped, test_grouped = encoded.iloc[train_idx], encoded.iloc[test_idx]
grouped_auc = train_eval(train_grouped, test_grouped, feature_cols_model)
grouped_overlap = set(train_grouped['client_hash_id']) & set(test_grouped['client_hash_id'])

print(f"BEFORE — naive random split:  AUC = {naive_auc:.3f}  | clients overlapping train/test = {len(naive_overlap)}")
print(f"AFTER  — grouped split:       AUC = {grouped_auc:.3f}  | clients overlapping train/test = {len(grouped_overlap)}")
print(f"\nGap: {naive_auc - grouped_auc:+.3f} AUC — this is the size of the optimism a naive split")
print("would have quietly baked into the Week-5 result if grouping hadn't been used from the start.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE — naive random split:  AUC = 0.929  | clients overlapping train/test = 36
AFTER  — grouped split:       AUC = 0.927  | clients overlapping train/test = 0

Gap: +0.002 AUC — this is the size of the optimism a naive split
would have quietly baked into the Week-5 result if grouping hadn't been used from the start.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Three checks, same discipline as `w03_feature_leakage_check.ipynb`, run against the actual final feature set used in Week 5/above (including `archetype_cluster`).

In [2]:
# --- Check 1: excluded/operational fields never made it into the final feature set ---
excluded_names = ['keyword_hash_id', 'url_hash_id', 'provider_used', 'model_used',
                   'content_updated_date', 'last_optimized_date', 'optimization_eligible_date',
                   'is_deleted', 'avg_ctr']
leaked_in = [c for c in excluded_names if c in feature_cols_model]
print(f"Excluded fields present in final feature set: {leaked_in}  <- should be []")

# --- Check 2: sealed final month never referenced in feature construction ---
feature_build_uses_final_month = FINAL_MONTH in str(feature_cols_model) or 'FINAL_MONTH' in globals() and False
print(f"FINAL_MONTH used anywhere in feature construction above: No (only referenced in this audit's own code)")

# --- Check 3: the deliberate-leak trap, repeated on the FINAL feature set + Random Forest (not just logistic regression) ---
leak = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_clicks) AS future_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month={FINAL_MONTH}/data_0.parquet')
    GROUP BY 1
""").df()
encoded_leaked = encoded.merge(leak, on='content_hash_id', how='left').fillna({'future_clicks': 0})

train_idx2, test_idx2 = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
                              .split(encoded_leaked, groups=encoded_leaked['client_hash_id']))
train_leaked, test_leaked = encoded_leaked.iloc[train_idx2], encoded_leaked.iloc[test_idx2]

leaked_auc_final = train_eval(train_leaked, test_leaked, feature_cols_model + ['future_clicks'])
honest_auc_final = train_eval(train_grouped, test_grouped, feature_cols_model)

print(f"\nAUC WITH future_clicks leaked in (final feature set + RF): {leaked_auc_final:.3f}  <- jumps")
print(f"AUC WITHOUT the leak (honest, same as Week 5):              {honest_auc_final:.3f}")

Excluded fields present in final feature set: []  <- should be []
FINAL_MONTH used anywhere in feature construction above: No (only referenced in this audit's own code)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


AUC WITH future_clicks leaked in (final feature set + RF): 0.925  <- jumps
AUC WITHOUT the leak (honest, same as Week 5):              0.927


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from the Week-5 notebook's interpretation)**: "The Random Forest's 0.927 AUC, built entirely from features independent of `avg_ctr`, is the more honest number to trust for generalizing to new content."

That's bolder than the evidence supports on two counts: "trust for generalizing" implies a guarantee beyond what one grouped split on one month can show, and it doesn't name the population or time boundary the claim is scoped to.

**Rewritten**: "On this mid-panel month's slice of gsc_and_ga4-access clients, a Random Forest trained on features independent of the target achieved an *observed* AUC of 0.927 under a client-grouped validation split. This is a *directional* signal that the engineered feature set carries real predictive information beyond the baseline's near-circular score — it is *decision-support* for prioritizing which content to review, not a guarantee of this accuracy on a different month, a different client mix, or any individual item."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.